# 16 — Multiple actions, persistence, and chat

For normal chat startup in a fresh kernel, run the project setup cell below, then **B. Live model setup**, **E. Chat setup**, and **F. Start or resume chat**. A, C and D are optional regression tests. B loads the model; no training is performed.

In an already-loaded kernel with a working session, rerun only F to continue chatting. Do not use Run All for routine startup: it executes the regression scenarios first.

Use the final source modules, including the corrected RequestAnalyzer and persistence/session modules. Select the Agent1 .venv kernel. Shut down another kernel holding the model before loading a second copy.

Scope: independent read-only order tracking and stock steps. Missing arguments are collected before execution. Order statuses/dates and stock quantities are demo fixtures. Historical experiments are preserved in the non-executable appendix.


In [ ]:
from pathlib import Path
import sys, os, subprocess, json
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside your Agent1 project.")
sys.path.insert(0, str(project_root / "src"))
print("Project:", project_root)

## A. Offline regression tests
This validates schema, control flow, tools, and generation inputs using controlled analyzer outputs. It does not measure the actual LLM.

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(project_root / "src") + os.pathsep + env.get("PYTHONPATH", "")
completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=project_root, env=env, capture_output=True, text=True,
)
print(completed.stdout)
print(completed.stderr)
assert completed.returncode == 0, "Fix test failures before running the live model."
# Test count depends on the source modules installed in your project.

## B. Live model setup
Use the same classifier path as notebook 15. If automatic discovery finds multiple files, set `CLASSIFIER_PATH` explicitly. This section loads the model only once.

In [ ]:
from pathlib import Path
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open the notebook inside your Agent1 project.")
sys.path.insert(0, str(project_root / "src"))
adapter_dir = project_root / "models/adapter_v2"
knowledge_path = project_root / "data/knowledge/demo_retail_knowledge_v1.json"
# Set this relative path only if you have multiple saved classifier files.
CLASSIFIER_PATH = None
if CLASSIFIER_PATH is None:
    candidates = sorted({*project_root.joinpath("models").rglob("*.joblib"), *project_root.joinpath("models").rglob("*.pkl")})
    if len(candidates) != 1:
        print("Saved model candidates:")
        for path in candidates: print(path.relative_to(project_root))
        raise ValueError("Set CLASSIFIER_PATH to your saved ensemble's relative path, then rerun this cell. No retraining is needed.")
    classifier_path = candidates[0]
else:
    classifier_path = project_root / CLASSIFIER_PATH
print("Classifier:", classifier_path)
print("Adapter:", adapter_dir)

In [ ]:
from agent1.classification.ensemble import IntentClassifierEnsemble
from agent1.routing.intent_router import IntentRouter
from agent1.routing.intent_mapping import INTENT_TO_ACTION
from agent1.analysis.request_analyzer import RequestAnalyzer
from agent1.conversation.context_builder import ContextBuilder
from agent1.rag.knowledge import KnowledgeStore
from agent1.generation.response_generator import ResponseGenerator
from agent1.generation.response_service import ResponseService
from agent1.agent import Agent

# Load only your own trusted saved classifier.
ensemble = IntentClassifierEnsemble.load(classifier_path)
# Leave the 4 GB GPU for Llama; sentence embeddings run on CPU.
ensemble.semantic_classifier.device = "cpu"
ensemble.semantic_classifier.embedding_model = None
knowledge = KnowledgeStore(knowledge_path)
generator = ResponseGenerator(adapter_dir)
responses = ResponseService(generator, knowledge)

def analyzer_llm(system_prompt, user_prompt):
    with generator.model.disable_adapter():
        return generator.generate(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            max_new_tokens=1024,
        )

agent = Agent(
    ensemble_model=ensemble,
    router=IntentRouter(INTENT_TO_ACTION),
    analyzer=RequestAnalyzer(
        llm=analyzer_llm,
        knowledge_intent_lookup=knowledge.select,
    ),
    context_builder=ContextBuilder(),
    response_service=responses,
)
print("Agent1 is ready. No training is performed.")

## C. Inspect both planning and execution
Each test resets the conversation. Assertions check actual tool records and argument association. Read the generated answer as well: it must cover both tasks, preserve IDs/statuses, and distinguish failed/unknown results from zero stock. Passing structural assertions alone does not establish answer quality.

If a live assertion fails, inspect the printed plan and results. Do not replace the analyzer output manually to turn a live failure into a pass.

In [ ]:
def say(text):
    print("You:", text)
    try:
        answer = agent.handle_message(text)
    except ValueError as exc:
        print("Analyzer/validation error:", exc)
        raise
    print("Agent1:", answer)
    print("Pending:", agent.state.has_pending_action())
    print("Pending decision:", agent.last_pending_decision)
    print("Response source:", responses.last_mode or "control flow")
    if agent.last_action_plan:
        print("Plan:", agent.last_action_plan.model_dump_json(indent=2))
    print("Step results:", json.dumps(agent.multi_action_runner.last_results, indent=2))
    return answer

def results():
    return agent.multi_action_runner.last_results

def require_pair(expected):
    actual = [(r["intent"], r["arguments"]) for r in results()]
    assert actual == expected, ("Wrong operations/argument association", actual)
    assert not agent.state.has_pending_action()
    print("Execution path: PASS. Review the answer above for grounding/completeness.")

In [ ]:
import inspect
print("Analyzer file:", inspect.getfile(type(agent.analyzer)))
print("Analyzer class:", type(agent.analyzer).__name__)
print("Knowledge lookup configured:", agent.analyzer.knowledge_intent_lookup is not None)
# Configuration diagnostics do not establish actual model accuracy.


### 1. Tracking plus stock
Expected: ORD-123 is shipped, ETA 2026-09-23; mock blue backpack quantity is 5.

In [ ]:
agent.reset()
say("Track order ORD-123 and check whether the blue backpack is available.")
require_pair([("track_order", {"order_id": "ORD-123"}),
              ("availability", {"product": "backpack", "color": "blue"})])
assert all(r["result"]["success"] for r in results())

### 2. Two orders, same service
Expected: ORD-123 shipped; ORD-456 processing. The model must keep both steps despite their sharing one service.

In [ ]:
agent.reset()
say("Track order ORD-123 and track order ORD-456.")
require_pair([("track_order", {"order_id": "ORD-123"}),
              ("track_order", {"order_id": "ORD-456"})])
assert [r["result"]["data"]["status"] for r in results()] == ["shipped", "processing"]

### 3. Missing order number
Expected: both steps retained; no tools execute before the reply. Then both execute once.

In [ ]:
agent.reset()
say("Track my order and check whether the blue backpack is available.")
assert agent.state.has_pending_action()
assert len(agent.state.pending_action_plan.steps) == 2
assert results() == []
say("ORD-123")
require_pair([("track_order", {"order_id": "ORD-123"}),
              ("availability", {"product": "backpack", "color": "blue"})])

### 4. Partial failure
Expected: unknown order reported; stock lookup still succeeds. The answer must not claim the order was tracked successfully.

In [ ]:
agent.reset()
say("Track order ORD-999 and check whether the blue backpack is available.")
require_pair([("track_order", {"order_id": "ORD-999"}),
              ("availability", {"product": "backpack", "color": "blue"})])
assert [r["result"]["success"] for r in results()] == [False, True]

### 5. Zero stock versus unknown inventory
The red backpack exists in the fixture with quantity 0. A blue jacket has no record and availability is unknown.

In [ ]:
for product, color, success in [("backpack", "red", True), ("jacket", "blue", False)]:
    agent.reset()
    say(f"Track order ORD-123 and check whether the {color} {product} is available.")
    require_pair([("track_order", {"order_id": "ORD-123"}),
                  ("availability", {"product": product, "color": color})])
    r = results()[1]["result"]
    assert r["success"] == success
    if success:
        assert r["data"]["quantity"] == 0
    else:
        assert "quantity" not in r["data"]

### 6. Pending-plan conversation control
Abandon and new request must discard both old tasks. An ambiguous reply must preserve the whole plan.

In [ ]:
def start_pending_pair():
    agent.reset()
    say("Track my order and check whether the blue backpack is available.")
    assert agent.state.has_pending_action()
    assert len(agent.state.pending_action_plan.steps) == 2
    assert results() == []

start_pending_pair()
say("Yes")
assert agent.last_pending_decision == "clarify"
assert agent.state.has_pending_action()
assert results() == []
say("Never mind")
assert agent.last_pending_decision == "abandon"
assert not agent.state.has_pending_action()
assert results() == []

start_pending_pair()
say("Actually, what time does Central close on Saturday?")
assert agent.last_pending_decision == "new_request"
assert not agent.state.has_pending_action()
assert results() == []

## D. Optional live SQLite checkpoint regression

The original test uses fresh agent instances sharing the loaded model. It writes a new demo conversation to SQLite. Skip it for routine chat startup.

In [ ]:
# Test of the history
# ```python
# Run in the notebook/kernel where your working `agent` is already loaded.
# No model load, training, or analyzer monkey-patching occurs here.
from pathlib import Path
from uuid import uuid4
from agent1.persistent_agent import PersistentAgent
from agent1.persistence.checkpoint_store import SQLiteCheckpointStore

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src/agent1").is_dir())
checkpoint_store = SQLiteCheckpointStore(project_root / "data/local/conversations.sqlite3")
template_agent = agent

def new_persistent_agent():
    return PersistentAgent(
        ensemble_model=template_agent.ensemble_model,
        router=template_agent.router,
        analyzer=template_agent.analyzer,
        context_builder=template_agent.context_builder,
        top_n_intents=template_agent.top_n_intents,
        response_service=template_agent.response_service,
        checkpoint_store=checkpoint_store,
    )

conversation_id = "live-resume-" + uuid4().hex
before = new_persistent_agent()
print("Before save:", before.handle_message(
    "Track my order and check whether the blue backpack is available."))
assert before.state.has_pending_action()
assert len(before.state.pending_action_plan.steps) == 2
assert before.state.get_missing_arguments() == ["0.order_id"]
assert before.multi_action_runner.last_results == []
before.save_conversation(conversation_id)

resumed = new_persistent_agent()
resumed.load_conversation(conversation_id)
assert resumed.state is resumed.action_runner.state
assert resumed.state is resumed.multi_action_runner.state
assert resumed.state.messages == before.state.messages
assert resumed.multi_action_runner.last_results == []
print("Restored missing:", resumed.state.get_missing_arguments())
print("After restore:", resumed.handle_message("ORD-123"))
records = resumed.multi_action_runner.last_results
assert [(r["intent"], r["arguments"]) for r in records] == [
    ("track_order", {"order_id": "ORD-123"}),
    ("availability", {"product": "backpack", "color": "blue"}),
]
assert all(r["result"]["success"] for r in records)
assert not resumed.state.has_pending_action()
# Save completion under the SAME ID, replacing the pending checkpoint.
resumed.save_conversation(conversation_id)

completed = new_persistent_agent().load_conversation(conversation_id)
assert not completed.state.has_pending_action()
assert completed.state.messages == resumed.state.messages
assert completed.multi_action_runner.last_results == []
print("SAVE / RESTORE / RESUME: PASS")
print("Review the answer: shipped ORD-123 and 5 blue backpacks (mock inventory).")
print("Saved conversation:", conversation_id)

# ```

## E. Chat setup — run once per session

SQLite is the default and uses the same local file as earlier tests. For Atlas, change BACKEND to "mongodb". Install the driver in this kernel if needed: %pip install "pymongo>=4.6,<5". The database user needs read/write access to agent1_demo, and Atlas must allow your current IP.

The MongoDB branch asks for the original database password in a hidden prompt and encodes it automatically. No environment variable is required. Never save the password in notebook source.

Backends have separate conversation lists; selecting MongoDB does not migrate SQLite conversations. Do not rerun this cell while a turn is unsaved: first use /retry-save, or explicitly discard it with /reload. Routine continuation needs only F.


In [ ]:
from getpass import getpass
from urllib.parse import quote_plus
from agent1.persistent_agent import PersistentAgent
from agent1.session import ChatSession
from agent1.chat_console import run_chat
from agent1.persistence.sqlite_session_store import SQLiteSessionStore

BACKEND = "sqlite"  # "sqlite" or "mongodb"
template_agent = agent

if BACKEND == "sqlite":
    store = SQLiteSessionStore(project_root / "data/local/conversations.sqlite3")
elif BACKEND == "mongodb":
    from agent1.persistence.mongo_store import MongoCheckpointStore
    uri = (
        "mongodb+srv://agent1_user:"
        + quote_plus(getpass("MongoDB password: "))
        + "@agent1.lpqwahh.mongodb.net/"
    )
    try:
        store = MongoCheckpointStore.from_uri(uri, database="agent1_demo")
    except Exception as exc:
        raise RuntimeError(
            f"MongoDB connection failed ({type(exc).__name__}). "
            "Check credentials and Atlas network access."
        ) from None
    finally:
        del uri
else:
    raise ValueError("BACKEND must be 'sqlite' or 'mongodb'.")

def make_agent(checkpoint_store):
    return PersistentAgent(
        ensemble_model=template_agent.ensemble_model,
        router=template_agent.router,
        analyzer=template_agent.analyzer,
        context_builder=template_agent.context_builder,
        top_n_intents=template_agent.top_n_intents,
        response_service=template_agent.response_service,
        checkpoint_store=checkpoint_store,
    )

session = ChatSession(store, make_agent)
print(f"Chat ready; backend={BACKEND}. Existing model reused.")


## F. Start or resume chat

Run the next cell, then type into its input prompt. After /quit, rerun the same cell to continue.

| Command | Purpose |
| --- | --- |
| /new | Start a separate conversation |
| /list | List saved conversations in this backend |
| /open ID | Restore a conversation and its pending request |
| /history | Show current messages |
| /retry-save | Retry a failed save without processing the turn again |
| /reload | Restore saved state, discarding unsaved progress |
| /quit | Stop chat input |

Try /new, then "Where is my order?", then "ORD-123".
For multi-action resume, ask "Track my order and check whether the blue backpack is available." When asked for the ID, use /list, record this conversation ID, and /quit. Run the chat cell again, /open that ID, and enter ORD-123. Expect shipped and 5 available, with the mock-inventory disclosure.

Other checks: ORD-999 should be unknown; "Never mind" abandons a pending request; "Actually, what time does Central close on Saturday?" replaces it with a store-hours question. Central's demo Saturday closing time is 18:00.

After a processing error, /quit and inspect a traceback using the diagnostic example in the appendix. /reload restores saved state; it does not fix the underlying cause. "Chat stopped. Only successfully saved turns are durable." is the normal exit message.


In [ ]:
run_chat(session)

## Completion and limits

Review grounding as well as execution assertions. Resume tests verify the selected backend only. Run the MongoDB comparison notebook or repeat the chat scenario with BACKEND="mongodb" to verify Atlas separately.

The interface is synchronous, with one caller/writer per conversation. Supported steps are independent and read-only; there is no dependency scheduling or automatic retry of model/tool execution. Single stock questions retain the earlier informational path; mock stock tools are connected to multi-action execution. The V2 adapter has not been retrained for multiple-result formatting.

Before committing, review git diff and exclude credentials and local database files. Saved outputs were cleared during cleanup so old runs are not presented as validation of this version.


## Appendix — historical experiments (not executed)

Prompt overrides, fallback wrappers, diagnostics and one-time reload code document how the implementation evolved. Active setup imports the final source implementation. These examples are reference material, not startup steps.

### Historical cell 10

```python
# # Replaces the analyzer’s prompt for this notebook session, leaving the execution and response-generation code unchanged.
# from agent1.routing.action_types import ActionType

# planning_prompt = """
# You plan actions for a retail assistant.
# Return ONLY a valid JSON object.

# Read the user's conversation first. Routing candidates are fallible
# suggestions, not additional requests. Never invent a return, refund,
# or other task merely because it appears in routing evidence.

# Distinguish:
# - Multiple explicitly requested tasks: multi_action.
# - One clear task: single_action.
# - Genuinely ambiguous user meaning: clarification.
# - A question requiring no tool: general_response.

# "And" joining two clear tasks means perform BOTH, not ask which one.
# Two orders to track require two separate steps.

# Executable operations:
# - track_order / order_service: arguments {"order_id": "..."}
# - availability / product_service:
#   arguments {"product": "...", "color": "..."}

# Use availability for checking stock.
# Extract arguments from the conversation; never invent missing values.
# Missing arguments do not make the requested operation ambiguous.
# Preserve other explicitly requested operations rather than dropping them.
# Mark dependencies using zero-based depends_on indices when necessary.

# Output fields:
# {
#   "request_type": "single_action|multi_action|clarification|general_response",
#   "intent": null,
#   "action": null,
#   "steps": [],
#   "candidate_intents": [],
#   "candidate_actions": [],
#   "needs_clarification": false,
#   "clarification_question": null,
#   "reason": "brief explanation"
# }

# For multi_action:
# - Put every requested task in steps.
# - Each step contains intent, action, arguments, depends_on, condition.
# - Use depends_on: [] and condition: null for independent tasks.
# - Leave top-level intent and action null.

# For single_action:
# - Set top-level intent and action.
# - Leave steps empty.

# For clarification:
# - Leave steps empty.
# - Set needs_clarification true and provide a specific question.
# - Include only alternatives actually supported by the conversation.

# For general_response:
# - Leave action null and steps empty.

# CONSISTENCY RULES:
# - If steps contains two or more operations, request_type MUST be
#   "multi_action", and top-level intent and action MUST both be null.
# - depends_on describes a required result from another operation,
#   NOT execution order. Independent lookups MUST have depends_on: [].
# - When color is extracted separately, product contains only the
#   product name: "green jacket" becomes product "jacket", color "green".

# Example:
# User: Track ORD-456 and check whether the green jacket is available.
# Output:
# {
#   "request_type": "multi_action",
#   "intent": null,
#   "action": null,
#   "steps": [
#     {
#       "intent": "track_order",
#       "action": "order_service",
#       "arguments": {"order_id": "ORD-456"},
#       "depends_on": [],
#       "condition": null
#     },
#     {
#       "intent": "availability",
#       "action": "product_service",
#       "arguments": {"product": "jacket", "color": "green"},
#       "depends_on": [],
#       "condition": null
#     }
#   ],
#   "candidate_intents": [],
#   "candidate_actions": [],
#   "needs_clarification": false,
#   "clarification_question": null,
#   "reason": "Two independent lookups."
# }

# A known operation with missing arguments is still a planned operation.
# Retain it in steps with empty arguments. The application collects missing
# arguments after planning. Do not drop other tasks in the same request.

# Example:
# User: Track my order and check the red jacket.
# Output:
# {
#   "request_type": "multi_action",
#   "intent": null,
#   "action": null,
#   "steps": [
#     {
#       "intent": "track_order",
#       "action": "order_service",
#       "arguments": {},
#       "depends_on": [],
#       "condition": null
#     },
#     {
#       "intent": "availability",
#       "action": "product_service",
#       "arguments": {"product": "jacket", "color": "red"},
#       "depends_on": [],
#       "condition": null
#     }
#   ],
#   "candidate_intents": [],
#   "candidate_actions": [],
#   "needs_clarification": false,
#   "clarification_question": null,
#   "reason": "Two requested operations; the order number must be collected."  
# }
# """
# planning_prompt += "\nAvailable services: " + ", ".join(
#     action.value for action in ActionType
# )

# def focused_analyzer_llm(system_prompt, user_prompt):
#     is_planning = "structured ActionPlan" in system_prompt

#     if is_planning:
#         effective_system = planning_prompt

#         # Keep the conversation; omit routing candidates and commentary.
#         conversation, separator, _ = user_prompt.partition(
#             "\n\nROUTING EVIDENCE"
#         )
#         if not separator:
#             raise ValueError("Cannot locate the routing-evidence section.")

#         effective_user = conversation.strip()
#     else:
#         # Preserve pending-reply classification.
#         effective_system = system_prompt
#         effective_user = user_prompt

#     # additional debug output for planning 
#     if is_planning:
#         print("ACTUAL PLANNING INPUT:\n", effective_user)

#     with generator.model.disable_adapter():
#         raw = generator.generate(
#             [
#                 {"role": "system", "content": effective_system},
#                 {"role": "user", "content": effective_user},
#             ],
#             max_new_tokens=1024,
#         )

#     print("Raw analyzer output:\n", raw)
#     return raw

# agent.analyzer.llm = focused_analyzer_llm
```

### Historical cell 22

```python
# Run AFTER applying the patch to source. Reuses generator, knowledge and agent.
# This replaces the analyzer instance, so no diagnostic wrappers remain active.
import importlib
import agent1.analysis.request_analyzer as analyzer_module

importlib.invalidate_caches()
importlib.reload(analyzer_module)
RequestAnalyzer = analyzer_module.RequestAnalyzer

previous_normalizer = getattr(agent.analyzer, 'request_type_normalizer', None)
agent.analyzer = RequestAnalyzer(
    llm=analyzer_llm,
    request_type_normalizer=previous_normalizer,
    knowledge_intent_lookup=knowledge.select,
)
agent.reset()
print('Source-backed analyzer ready; existing model reused.')
```

### Historical cell 24

```python
# agent.reset()
# say("What time does Central close on Saturday?")
```

### Historical cell 25

```python
# question = "What time does Central close on Saturday?"
# intent = "store_opening_hours"

# context = responses.knowledge_store.get_context(intent)
# print("Retrieved knowledge:\n", context or "(none)")

# if context:
#     answer = responses.respond(question, intent=intent)
#     print("\nAnswer:", answer)
#     print("Response source:", responses.last_mode)
```

### Historical cell 26

```python
# # test
# # Run once; keeps the already-loaded model.
# original_analyze = agent.analyzer.analyze

# def debug_analyze(
#     conversation_context,
#     routing_decision,
#     intent_candidates_text,
# ):
#     print("Router intent:", routing_decision.intent)
#     print("Intent score:", routing_decision.intent_score)
#     print("Low confidence:", routing_decision.low_confidence)
#     print("Intent candidates:\n", intent_candidates_text)

#     return original_analyze(
#         conversation_context,
#         routing_decision,
#         intent_candidates_text,
#     )

# agent.analyzer.analyze = debug_analyze
```

### Historical cell 27

```python
# agent.reset()
# say("What time does Central close on Saturday?")
```

### Historical cell 28

```python
# # Restore the method saved before the diagnostic wrapper.
# agent.analyzer.analyze = original_analyze
# base_analyze = agent.analyzer.analyze

# def analyze_with_knowledge_fallback(
#     conversation_context,
#     routing_decision,
#     intent_candidates_text,
# ):
#     plan = base_analyze(
#         conversation_context,
#         routing_decision,
#         intent_candidates_text,
#     )

#     request_type = getattr(plan.request_type, "value", plan.request_type)
#     candidate = routing_decision.intent

#     if (
#         request_type == "general_response"
#         and plan.intent is None
#         and not plan.needs_clarification
#         and not routing_decision.low_confidence
#         and responses.knowledge_store.select(candidate)
#     ):
#         data = plan.model_dump()
#         data["intent"] = candidate
#         plan = type(plan).model_validate(data)
#         print("Knowledge-intent fallback:", candidate)

#     return plan

# agent.analyzer.analyze = analyze_with_knowledge_fallback
```

### Historical cell 29

```python
# # final test
# # Start a two-task request with a missing order number.
# agent.reset()
# say("Track my order and check whether the blue backpack is available.")

# assert agent.state.has_pending_action()
# assert len(agent.state.pending_action_plan.steps) == 2
# assert results() == []

# # Replace the pending request with a store-hours question.
# say("Actually, what time does Central close on Saturday?")

# assert not agent.state.has_pending_action()
# assert agent.last_pending_decision == "new_request"
# assert results() == []
# assert agent.last_action_plan.intent == "store_opening_hours"
# assert responses.last_mode == "v2"

# print("Topic-change execution path: PASS")
```

### Historical cell 33

```python
# trace
import traceback

try:
    session.reload_saved()
    answer = session.send(
        "Where is my order?"
    )
    print(answer)
except Exception:
    traceback.print_exc()
```

### Historical cell 34

```python
# refreshes the analyzer while reusing your loaded model
import importlib
import agent1.analysis.request_analyzer as analyzer_module

importlib.reload(analyzer_module)
RequestAnalyzer = analyzer_module.RequestAnalyzer

old_analyzer = session.agent.analyzer

updated_analyzer = RequestAnalyzer(
    llm=old_analyzer.llm,
    request_type_normalizer=old_analyzer.request_type_normalizer,
    knowledge_intent_lookup=old_analyzer.knowledge_intent_lookup,
)

# Update both the active session and the template for future sessions.
agent.analyzer = updated_analyzer
template_agent.analyzer = updated_analyzer
session.agent.analyzer = updated_analyzer

session.reload_saved()
print("Analyzer updated; saved conversation restored.")
```

### Historical cell 35

```python
run_chat(session)
# /new
# Where is my order?
# ORD-123
# /quit
```

### Earlier model callback

```python
# def analyzer_llm(system_prompt, user_prompt):
#     # V2 was trained to answer grounded questions, not to emit ActionPlan JSON.
#     with generator.model.disable_adapter():
#         return generator.generate([
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": user_prompt},
#         ], max_new_tokens=1024)

```